In [81]:
import pandas as pd

In [82]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"
columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount', 'total_amount']
df = pd.read_parquet(url, columns=columns)  # .head(5)
df.head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [83]:
row = df.iloc[0]
print(row)
# df.loc[0, 'passenger_count'] = pd.NA

lpep_pickup_datetime     2025-10-01 00:21:47
lpep_dropoff_datetime    2025-10-01 00:24:37
PULocationID                             247
DOLocationID                              69
passenger_count                          1.0
trip_distance                            0.7
tip_amount                               1.7
total_amount                            10.0
Name: 0, dtype: object


In [84]:
from dataclasses import dataclass

@dataclass
class Ride:
    lpep_pickup_datetime: str  
    lpep_dropoff_datetime: str
    PULocationID: int
    DOLocationID: int
    passenger_count: float
    trip_distance: float
    tip_amount: float
    total_amount: float


In [85]:
def ride_from_row(row):
    return Ride(     
        lpep_pickup_datetime=row['lpep_pickup_datetime'].strftime('%Y-%m-%d %H:%M:%S'),
        lpep_dropoff_datetime=row['lpep_dropoff_datetime'].strftime('%Y-%m-%d %H:%M:%S'),
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        passenger_count=float(row['passenger_count']),
        trip_distance=float(row['trip_distance']),
        tip_amount=float(row['tip_amount']),
        total_amount=float(row['total_amount']),
    )

In [86]:
ride = ride_from_row(df.iloc[0])
ride

Ride(lpep_pickup_datetime='2025-10-01 00:21:47', lpep_dropoff_datetime='2025-10-01 00:24:37', PULocationID=247, DOLocationID=69, passenger_count=1.0, trip_distance=0.7, tip_amount=1.7, total_amount=10.0)

In [87]:
import json
from kafka import KafkaProducer

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

In [88]:
import dataclasses

topic_name = 'green-trips'

def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    # print(f"Before serialization: {ride_dict}")
    ride_dict = {k: (None if isinstance(v, float) and v != v else v)
          for k, v in ride_dict.items()}
    # print(f"After serialization: {ride_dict}")

    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')

In [89]:
import json
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [ ]:
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    # print(f"Sent: {ride}")
    # time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

Sent: Ride(lpep_pickup_datetime='2025-10-01 00:21:47', lpep_dropoff_datetime='2025-10-01 00:24:37', PULocationID=247, DOLocationID=69, passenger_count=1.0, trip_distance=0.7, tip_amount=1.7, total_amount=10.0)
Sent: Ride(lpep_pickup_datetime='2025-10-01 00:14:03', lpep_dropoff_datetime='2025-10-01 00:24:14', PULocationID=66, DOLocationID=25, passenger_count=1.0, trip_distance=1.61, tip_amount=2.78, total_amount=16.68)
Sent: Ride(lpep_pickup_datetime='2025-10-01 00:16:44', lpep_dropoff_datetime='2025-10-01 00:16:47', PULocationID=244, DOLocationID=244, passenger_count=1.0, trip_distance=0.0, tip_amount=2.2, total_amount=13.2)
Sent: Ride(lpep_pickup_datetime='2025-10-01 00:07:36', lpep_dropoff_datetime='2025-10-01 00:32:14', PULocationID=95, DOLocationID=170, passenger_count=1.0, trip_distance=10.37, tip_amount=11.31, total_amount=67.85)
Sent: Ride(lpep_pickup_datetime='2025-09-30 21:10:29', lpep_dropoff_datetime='2025-09-30 21:22:30', PULocationID=82, DOLocationID=138, passenger_count=1